# 02 - Train and Evaluate MLPDeconv

This notebook loads saved pseudobulks, trains the PyTorch MLP with L1 loss and early stopping, evaluates held-out pseudobulks, creates diagnostic plots, and saves the model checkpoint.

In [34]:
from __future__ import annotations

import copy
import importlib.util
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Sequence

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "torch": "torch",
}
missing_packages = [name for name, module in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    raise ImportError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". Install the project environment first, for example with `uv sync`."
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root, where pyproject.toml is located.")

REFERENCE_LABEL = "rat_sub_std"

REFERENCE_DIR = PROJECT_ROOT / "data" / "references" / REFERENCE_LABEL
PSEUDOBULK_DIR = REFERENCE_DIR / "pseudobulk"
PLOTS_DIR = PSEUDOBULK_DIR / "plots"
MODEL_DIR = REFERENCE_DIR / "model"

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

CELLTYPE_COLORS = {
    "neurons": "#D35400",
    "exc_neurons": "#E67E22",
    "inh_neurons": "#F39C12",
    "M/M": "#2E8B57",
    "microglia": "#006D5B",
    "macrophages": "#455A64",
    "hom_mg": "#00A087",
    "inf_mg": "#91D1C2",
    "prol_mg": "#008280",
    "prol_infil_mg": "#4DBBD5",
    "oligodendrocytes": "#5E548E",
    "opc": "#9F86C0",
    "astrocytes": "#BE95C4",
    "endothelial_cells": "#2B4162",
    "epithelial_cells": "#4A6FA5",
    "fibroblasts": "#D6CCC2",
    "b_cells": "#631879",
    "t_cells": "#A20056",
}

CELLTYPE_ORDER = [
    "exc_neurons",
    "inh_neurons",
    "neurons",
    "M/M",
    "hom_mg",
    "inf_mg",
    "prol_mg",
    "prol_infil_mg",
    "microglia",
    "macrophages",
    "oligodendrocytes",
    "opc",
    "astrocytes",
    "endothelial_cells",
    "epithelial_cells",
    "fibroblasts",
    "b_cells",
    "t_cells",
]
DEFAULT_CELLTYPE_COLOR = "#9E9E9E"


def standardize_celltype_label(cell_type: str) -> str:
    key = str(cell_type).strip()
    key = key.replace("-", "_").replace(" ", "_")
    key = "_".join(part for part in key.split("_") if part)
    lower_key = key.lower()
    aliases = {
        "m/m": "M/M",
        "m_m": "M/M",
        "mm": "M/M",
        "b_cell": "b_cells",
        "b_cells": "b_cells",
        "t_cell": "t_cells",
        "t_cells": "t_cells",
        "opc": "opc",
        "opcs": "opc",
        "astrocyte": "astrocytes",
        "astrocytes": "astrocytes",
        "microglia": "microglia",
        "macrophage": "macrophages",
        "macrophages": "macrophages",
        "endothelial_cell": "endothelial_cells",
        "endothelial_cells": "endothelial_cells",
        "epithelial_cell": "epithelial_cells",
        "epithelial_cells": "epithelial_cells",
        "fibroblast": "fibroblasts",
        "fibroblasts": "fibroblasts",
        "neuron": "neurons",
        "neurons": "neurons",
    }
    return aliases.get(lower_key, lower_key)


def order_celltypes(cell_types: Sequence[str]) -> list[str]:
    order_rank = {cell_type: idx for idx, cell_type in enumerate(CELLTYPE_ORDER)}
    return sorted(
        [str(cell_type) for cell_type in cell_types],
        key=lambda cell_type: (
            order_rank.get(standardize_celltype_label(cell_type), len(order_rank)),
            standardize_celltype_label(cell_type),
            str(cell_type),
        ),
    )


def celltype_colors_for(cell_types: Sequence[str]) -> dict[str, str]:
    return {
        str(cell_type): CELLTYPE_COLORS.get(standardize_celltype_label(cell_type), DEFAULT_CELLTYPE_COLOR)
        for cell_type in cell_types
    }


Device: cuda


## Load saved pseudobulks


In [35]:
def load_pseudobulk_dataset(npz_path: str | Path) -> dict:
    path = Path(npz_path)
    if not path.exists():
        raise FileNotFoundError(f"Missing pseudobulk dataset: {path}")
    with np.load(path, allow_pickle=True) as data:
        return {
            "X": data["X"].astype(np.float32),
            "y": data["y"].astype(np.float32),
            "genes": data["genes"].astype(str).tolist(),
            "cell_types": data["cell_types"].astype(str).tolist(),
            "sample_ids": data["sample_ids"].astype(str).tolist(),
        }


def reorder_targets_to_celltype_order(dataset: dict, ordered_cell_types: Sequence[str]) -> dict:
    original_cell_types = dataset["cell_types"]
    column_index = [original_cell_types.index(cell_type) for cell_type in ordered_cell_types]
    dataset = dataset.copy()
    dataset["y"] = dataset["y"][:, column_index]
    dataset["cell_types"] = list(ordered_cell_types)
    return dataset


train_pseudobulk = load_pseudobulk_dataset(PSEUDOBULK_DIR / "train_pseudobulk.npz")
test_pseudobulk = load_pseudobulk_dataset(PSEUDOBULK_DIR / "test_pseudobulk.npz")

CELL_TYPES = order_celltypes(train_pseudobulk["cell_types"])
CELLTYPE_PLOT_COLORS = celltype_colors_for(CELL_TYPES)
train_pseudobulk = reorder_targets_to_celltype_order(train_pseudobulk, CELL_TYPES)
test_pseudobulk = reorder_targets_to_celltype_order(test_pseudobulk, CELL_TYPES)

train_pb_X = train_pseudobulk["X"]
train_pb_y = train_pseudobulk["y"]
train_sample_ids = train_pseudobulk["sample_ids"]
test_pb_X = test_pseudobulk["X"]
test_pb_y = test_pseudobulk["y"]
test_sample_ids = test_pseudobulk["sample_ids"]
genes = train_pseudobulk["genes"]

print(f"Train pseudobulks: {train_pb_X.shape}")
print(f"Test pseudobulks: {test_pb_X.shape}")
display(pd.DataFrame({"celltype": CELL_TYPES, "color": [CELLTYPE_PLOT_COLORS[ct] for ct in CELL_TYPES]}))


Train pseudobulks: (10000, 13685)
Test pseudobulks: (2500, 13685)


,celltype,color
0,exc_neurons,#E67E22
1,inh_neurons,#F39C12
2,hom_mg,#00A087
3,inf_mg,#91D1C2
4,prol_mg,#008280
5,macrophages,#455A64
6,oligodendrocytes,#5E548E
7,opc,#9F86C0
8,astrocytes,#BE95C4
9,endothelial_cells,#2B4162


## Normalize and standardize

The gene-wise mean and standard deviation are fitted only on the training pseudobulk split, then applied to validation, test, and future real bulk data. This is the standard no-leakage pattern: validation/test data may be transformed with training-set parameters, but their own distribution is not used to fit preprocessing.


In [36]:
def normalize_bulk_counts(X: np.ndarray, scale_factor: float = 1_000_000.0) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    library_sizes = X.sum(axis=1, keepdims=True)
    if np.any(library_sizes <= 0):
        raise ValueError("At least one bulk sample has zero total counts and cannot be normalized.")
    normalized = (X / library_sizes) * scale_factor
    return np.log1p(normalized).astype(np.float32)


def split_train_validation(
    X: np.ndarray,
    y: np.ndarray,
    validation_fraction: float = 0.15,
    seed: int = RANDOM_SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if X.shape[0] != y.shape[0]:
        raise ValueError("X and y must have the same number of samples.")
    if X.shape[0] < 2:
        raise ValueError("Need at least two pseudobulks to create a validation split.")
    rng = np.random.default_rng(seed)
    indices = rng.permutation(X.shape[0])
    n_validation = int(round(X.shape[0] * validation_fraction))
    n_validation = max(1, min(X.shape[0] - 1, n_validation))
    validation_idx = indices[:n_validation]
    train_idx = indices[n_validation:]
    return X[train_idx], y[train_idx], X[validation_idx], y[validation_idx]


def fit_standardizer(X_train: np.ndarray, eps: float = 1e-6) -> tuple[np.ndarray, np.ndarray]:
    mean = X_train.mean(axis=0).astype(np.float32)
    std = X_train.std(axis=0).astype(np.float32)
    std[std < eps] = 1.0
    return mean, std


def apply_standardizer(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((X - mean) / std).astype(np.float32)


In [37]:
train_features_all = normalize_bulk_counts(train_pb_X)
test_features = normalize_bulk_counts(test_pb_X)
X_train_raw, y_train, X_val_raw, y_val = split_train_validation(
    train_features_all, train_pb_y, validation_fraction=0.15, seed=RANDOM_SEED
)
gene_mean, gene_std = fit_standardizer(X_train_raw)
X_train = apply_standardizer(X_train_raw, gene_mean, gene_std)
X_val = apply_standardizer(X_val_raw, gene_mean, gene_std)
X_test = apply_standardizer(test_features, gene_mean, gene_std)
y_test = test_pb_y.astype(np.float32)
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")


X_train: (8500, 13685), y_train: (8500, 14)
X_val:   (1500, 13685), y_val:   (1500, 14)
X_test:  (2500, 13685), y_test:  (2500, 14)


## Train the PyTorch MLP


In [38]:
def set_reproducible_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class BulkFractionDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[index], self.y[index]


class MLPDeconvRegressor(nn.Module):
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        hidden_sizes: Sequence[int] = (512, 256, 128),
        dropout: float = 0.2,
    ):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_sizes:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        logits = self.network(X)
        return torch.softmax(logits, dim=1)

@dataclass
class TrainingConfig:
    hidden_sizes: tuple[int, ...] = (1024, 512, 256)
    dropout: float = 0.0
    batch_size: int = 256
    learning_rate: float = 0.0011076771216192182
    weight_decay: float = 1.3745653445317599e-06
    max_steps: int = 5000
    validation_every: int = 100
    patience: int = 15
    min_delta: float = 1e-5
    loss_name: str = "MSELoss"

In [39]:
def mean_epoch_loss(model: nn.Module, loader: DataLoader, loss_fn: nn.Module, device: torch.device) -> float:
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            loss = loss_fn(model(X_batch), y_batch)
            batch_size = X_batch.shape[0]
            total_loss += float(loss.item()) * batch_size
            total_samples += batch_size
    return total_loss / max(total_samples, 1)


def train_mlp_regressor(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    config: TrainingConfig,
    device: torch.device = DEVICE,
    seed: int = RANDOM_SEED,
) -> tuple[MLPDeconvRegressor, pd.DataFrame]:
    if config.max_steps <= 0:
        raise ValueError("max_steps must be positive.")
    if config.validation_every <= 0:
        raise ValueError("validation_every must be positive.")

    set_reproducible_seed(seed)
    train_loader = DataLoader(BulkFractionDataset(X_train, y_train), batch_size=config.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(BulkFractionDataset(X_val, y_val), batch_size=config.batch_size, shuffle=False, num_workers=0)
    model = MLPDeconvRegressor(
        input_dim=X_train.shape[1],
        output_dim=y_train.shape[1],
        hidden_sizes=config.hidden_sizes,
        dropout=config.dropout,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    loss_fn = nn.MSELoss()
    best_val_loss = math.inf
    best_state = copy.deepcopy(model.state_dict())
    checks_without_improvement = 0
    history = []
    global_step = 0
    epoch = 0
    running_loss_sum = 0.0
    running_samples = 0
    stop_training = False

    while global_step < config.max_steps and not stop_training:
        epoch += 1
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

            batch_size = X_batch.shape[0]
            global_step += 1
            running_loss_sum += float(loss.item()) * batch_size
            running_samples += batch_size

            should_validate = (
                global_step == 1
                or global_step % config.validation_every == 0
                or global_step == config.max_steps
            )
            if should_validate:
                train_loss = running_loss_sum / max(running_samples, 1)
                val_loss = mean_epoch_loss(model, val_loader, loss_fn, device)
                history.append({"step": global_step, "epoch": epoch, "train_MSE": train_loss, "val_MSE": val_loss})
                print(f"Step {global_step:05d} | epoch={epoch:03d} | train_MSE={train_loss:.6f} | val_MSE={val_loss:.6f}")
                running_loss_sum = 0.0
                running_samples = 0

                if val_loss < best_val_loss - config.min_delta:
                    best_val_loss = val_loss
                    best_state = copy.deepcopy(model.state_dict())
                    checks_without_improvement = 0
                else:
                    checks_without_improvement += 1
                    if checks_without_improvement >= config.patience:
                        print(
                            f"Early stopping at step {global_step}. "
                            f"Best validation MSE loss: {best_val_loss:.6f}"
                        )
                        stop_training = True
                        break

            if global_step >= config.max_steps:
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


In [40]:

# # rat_main
# training_config = TrainingConfig(
#     hidden_sizes=(1024, 512, 256),
#     dropout=0.0,
#     batch_size=256,
#     learning_rate=0.0011076771216192182,
#     weight_decay=1.3745653445317599e-06,
#     max_steps=5000,
#     validation_every=100,
#     patience=15,
#     min_delta=1e-5,
#     loss_name="MSELoss",
# )

# rat_sub
training_config = TrainingConfig(
    hidden_sizes=(1024,),
    dropout=0.1,
    batch_size=256,
    learning_rate=0.0003278187653397617,
    weight_decay=4.982752357076448e-07,
    max_steps=5000,
    validation_every=100,
    patience=15,
    min_delta=1e-5,
    loss_name="MSELoss",
)

# # mouse_main
# training_config = TrainingConfig(
#     hidden_sizes=(2048, 1024),
#     dropout=0.0,
#     batch_size=256,
#     learning_rate=0.0004497099260523556,
#     weight_decay=0.0016642218944802722,
#     max_steps=5000,
#     validation_every=100,
#     patience=15,
#     min_delta=1e-5,
#     loss_name="MSELoss",
# )

# # mouse_sub
# training_config = TrainingConfig(
#     hidden_sizes=(1024,),
#     dropout=0.1,
#     batch_size=256,
#     learning_rate=0.0003278187653397617,
#     weight_decay=4.982752357076448e-07,
#     max_steps=5000,
#     validation_every=100,
#     patience=15,
#     min_delta=1e-5,
#     loss_name="MSELoss",
# )

model, training_history = train_mlp_regressor(
    X_train, y_train, X_val, y_val, config=training_config, device=DEVICE, seed=RANDOM_SEED
)
training_history.tail()


Step 00001 | epoch=001 | train_MSE=0.011325 | val_MSE=0.018427
Step 00100 | epoch=003 | train_MSE=0.011831 | val_MSE=0.006121
Step 00200 | epoch=006 | train_MSE=0.006884 | val_MSE=0.007045
Step 00300 | epoch=009 | train_MSE=0.005633 | val_MSE=0.004258
Step 00400 | epoch=012 | train_MSE=0.004182 | val_MSE=0.002893
Step 00500 | epoch=015 | train_MSE=0.002900 | val_MSE=0.001910
Step 00600 | epoch=018 | train_MSE=0.001924 | val_MSE=0.001270
Step 00700 | epoch=021 | train_MSE=0.001012 | val_MSE=0.000789
Step 00800 | epoch=024 | train_MSE=0.000611 | val_MSE=0.000468
Step 00900 | epoch=027 | train_MSE=0.000416 | val_MSE=0.000404
Step 01000 | epoch=030 | train_MSE=0.000295 | val_MSE=0.000292
Step 01100 | epoch=033 | train_MSE=0.000227 | val_MSE=0.000301
Step 01200 | epoch=036 | train_MSE=0.000194 | val_MSE=0.000252
Step 01300 | epoch=039 | train_MSE=0.000160 | val_MSE=0.000237
Step 01400 | epoch=042 | train_MSE=0.000150 | val_MSE=0.000240
Step 01500 | epoch=045 | train_MSE=0.000146 | val_MSE=0

,step,epoch,train_MSE,val_MSE
46,4600,136,0.000068,0.000180
47,4700,139,0.000094,0.000177
48,4800,142,0.000091,0.000157
49,4900,145,0.000067,0.000156
50,5000,148,0.000084,0.000170


## 7. Evaluate on held-out pseudobulks

The test set is generated from held-out cells. Metrics are now RMSE, Pearson correlation, and R2. They are reported globally, per cell type, and per pseudobulk sample.

For visual inspection, the notebook selects 8 representative pseudobulks by RMSE, plots real vs predicted stacked proportions, and then plots the best and worst predicted cell types.


In [41]:
@torch.no_grad()
def predict_array(model: nn.Module, X: np.ndarray, batch_size: int = 256, device: torch.device = DEVICE) -> np.ndarray:
    model.eval()
    tensor_dataset = torch.as_tensor(X, dtype=torch.float32)
    loader = DataLoader(tensor_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    predictions = []
    for X_batch in loader:
        X_batch = X_batch.to(device)
        predictions.append(model(X_batch).cpu().numpy())
    return np.vstack(predictions).astype(np.float32)


def rmse_score(y_true: np.ndarray, y_pred: np.ndarray, axis=None) -> np.ndarray:
    return np.sqrt(np.mean(np.square(y_pred - y_true), axis=axis))


def pearson_score(y_true: np.ndarray, y_pred: np.ndarray, axis=None, eps: float = 1e-12) -> np.ndarray:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if axis is None:
        y_true = y_true.ravel()
        y_pred = y_pred.ravel()
        true_std = y_true.std()
        pred_std = y_pred.std()
        if true_std < eps or pred_std < eps:
            return np.nan
        return float(np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean())) / (true_std * pred_std))

    true_mean = np.mean(y_true, axis=axis, keepdims=True)
    pred_mean = np.mean(y_pred, axis=axis, keepdims=True)
    centered_true = y_true - true_mean
    centered_pred = y_pred - pred_mean
    numerator = np.mean(centered_true * centered_pred, axis=axis)
    denominator = np.std(y_true, axis=axis) * np.std(y_pred, axis=axis)
    return np.divide(numerator, denominator, out=np.full_like(numerator, np.nan, dtype=np.float64), where=denominator >= eps)


def r2_score(y_true: np.ndarray, y_pred: np.ndarray, axis=None, eps: float = 1e-12) -> np.ndarray:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = np.sum(np.square(y_true - y_pred), axis=axis)
    ss_tot = np.sum(np.square(y_true - np.mean(y_true, axis=axis, keepdims=True)), axis=axis)
    return np.divide(1 - (ss_res / ss_tot), 1.0, out=np.full_like(ss_res, np.nan, dtype=np.float64), where=ss_tot >= eps)


def deconvolution_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    cell_types: Sequence[str],
    sample_ids: Sequence[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch: y_true={y_true.shape}, y_pred={y_pred.shape}")

    if sample_ids is None:
        sample_ids = [f"sample_{idx:05d}" for idx in range(y_true.shape[0])]

    per_celltype = pd.DataFrame(
        {
            "celltype": list(cell_types),
            "rmse": rmse_score(y_true, y_pred, axis=0),
            "pearson": pearson_score(y_true, y_pred, axis=0),
            "r2": r2_score(y_true, y_pred, axis=0),
        }
    ).sort_values("rmse", ascending=True).reset_index(drop=True)

    per_sample = pd.DataFrame(
        {
            "sample_id": list(sample_ids),
            "rmse": rmse_score(y_true, y_pred, axis=1),
            "pearson": pearson_score(y_true, y_pred, axis=1),
            "r2": r2_score(y_true, y_pred, axis=1),
        }
    ).sort_values("rmse", ascending=True).reset_index(drop=True)

    global_metrics = pd.Series(
        {
            "rmse": float(rmse_score(y_true, y_pred)),
            "pearson": float(pearson_score(y_true, y_pred)),
            "r2": float(r2_score(y_true, y_pred)),
        },
        name="metrics",
    )
    return per_celltype, per_sample, global_metrics


def select_example_samples(per_sample_metrics: pd.DataFrame, n_examples: int = 8) -> list[str]:
    metrics = per_sample_metrics.sort_values("rmse", ascending=True).reset_index(drop=True)
    if len(metrics) <= n_examples:
        return metrics["sample_id"].astype(str).tolist()

    n_best = n_examples // 2
    n_worst = n_examples - n_best
    selected = pd.concat([metrics.head(n_best), metrics.tail(n_worst)], axis=0)
    return selected["sample_id"].astype(str).tolist()


def plot_real_vs_predicted_stacked_bars(
    y_true_df: pd.DataFrame,
    y_pred_df: pd.DataFrame,
    sample_ids: Sequence[str],
    celltype_colors: dict[str, str],
    output_path: str | Path,
    title: str,
) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sample_ids = [str(sample_id) for sample_id in sample_ids]
    cell_types = list(y_true_df.columns)

    x_positions = np.arange(len(sample_ids), dtype=float)
    width = 0.32
    pair_gap = 0.06
    real_positions = x_positions - (width + pair_gap) / 2
    pred_positions = x_positions + (width + pair_gap) / 2
    real_bottom = np.zeros(len(sample_ids), dtype=float)
    pred_bottom = np.zeros(len(sample_ids), dtype=float)

    fig_width = max(10, len(sample_ids) * 1.25)
    fig, ax = plt.subplots(figsize=(fig_width, 5.5))
    for cell_type in cell_types:
        color = celltype_colors.get(cell_type, DEFAULT_CELLTYPE_COLOR)
        real_values = y_true_df.loc[sample_ids, cell_type].to_numpy(dtype=float)
        pred_values = y_pred_df.loc[sample_ids, cell_type].to_numpy(dtype=float)
        ax.bar(real_positions, real_values, width=width, bottom=real_bottom, color=color, edgecolor="white", linewidth=0.4)
        ax.bar(pred_positions, pred_values, width=width, bottom=pred_bottom, color=color, edgecolor="white", linewidth=0.4)
        real_bottom += real_values
        pred_bottom += pred_values

    ax.set_title(title)
    ax.set_ylabel("Proportion")
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(sample_ids, rotation=45, ha="right")
    ax.text(0.01, 0.98, "left bar: real    right bar: predicted", transform=ax.transAxes, va="top")
    handles = [plt.Rectangle((0, 0), 1, 1, color=celltype_colors.get(ct, DEFAULT_CELLTYPE_COLOR)) for ct in cell_types]
    ax.legend(handles, cell_types, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, title="Cell type")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved stacked real-vs-predicted plot to {output_path}")
    return output_path


def plot_best_worst_celltype_predictions(
    y_true_df: pd.DataFrame,
    y_pred_df: pd.DataFrame,
    per_celltype_metrics: pd.DataFrame,
    celltype_colors: dict[str, str],
    output_path: str | Path,
    n_best: int = 3,
    n_worst: int = 3,
) -> tuple[Path, pd.DataFrame]:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    ordered = per_celltype_metrics.sort_values("rmse", ascending=True)
    selected = pd.concat([ordered.head(n_best).assign(group="best"), ordered.tail(n_worst).assign(group="worst")], axis=0)
    selected = selected.drop_duplicates("celltype").reset_index(drop=True)

    n_panels = len(selected)
    fig, axes = plt.subplots(1, n_panels, figsize=(max(4 * n_panels, 8), 4), sharex=True, sharey=True)
    if n_panels == 1:
        axes = [axes]

    for ax, row in zip(axes, selected.itertuples(index=False)):
        cell_type = row.celltype
        color = celltype_colors.get(cell_type, DEFAULT_CELLTYPE_COLOR)
        ax.scatter(y_true_df[cell_type], y_pred_df[cell_type], s=20, alpha=0.65, color=color)
        ax.plot([0, 1], [0, 1], color="#333333", linewidth=1, linestyle="--")
        ax.set_title(f"{row.group}: {cell_type}\nRMSE={row.rmse:.3f}, r={row.pearson:.3f}, R2={row.r2:.3f}")
        ax.set_xlabel("Real proportion")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    axes[0].set_ylabel("Predicted proportion")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved best/worst celltype prediction plot to {output_path}")
    return output_path, selected


In [42]:
test_predictions = predict_array(model, X_test, batch_size=256, device=DEVICE)
test_true_df = pd.DataFrame(y_test, index=test_sample_ids, columns=CELL_TYPES)
test_pred_df = pd.DataFrame(test_predictions, index=test_sample_ids, columns=CELL_TYPES)

per_celltype_metrics, per_sample_metrics, global_metrics = deconvolution_metrics(
    y_test,
    test_predictions,
    CELL_TYPES,
    sample_ids=test_sample_ids,
)

display(global_metrics)
display(per_celltype_metrics)
display(per_sample_metrics)

example_sample_ids = select_example_samples(per_sample_metrics, n_examples=16)
example_metrics = per_sample_metrics.set_index("sample_id").loc[example_sample_ids, ["rmse", "pearson", "r2"]]
display(example_metrics)

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
example_plot_path = plot_real_vs_predicted_stacked_bars(
    test_true_df,
    test_pred_df,
    sample_ids=example_sample_ids,
    celltype_colors=CELLTYPE_PLOT_COLORS,
    output_path=PLOTS_DIR / "test_8_pseudobulk_examples_real_vs_predicted_stacked.png",
    title="Held-out pseudobulk examples: real vs predicted proportions",
)
best_worst_plot_path, best_worst_celltypes = plot_best_worst_celltype_predictions(
    test_true_df,
    test_pred_df,
    per_celltype_metrics,
    celltype_colors=CELLTYPE_PLOT_COLORS,
    output_path=PLOTS_DIR / "test_best_worst_celltype_predictions.png",
    n_best=3,
    n_worst=3,
)
display(best_worst_celltypes)

PSEUDOBULK_DIR.mkdir(parents=True, exist_ok=True)
per_celltype_metrics.to_csv(PSEUDOBULK_DIR / "test_metrics_by_celltype.csv", index=False)
per_sample_metrics.to_csv(PSEUDOBULK_DIR / "test_metrics_by_sample.csv", index=False)
global_metrics.to_frame().to_csv(PSEUDOBULK_DIR / "test_metrics_global.csv")
example_metrics.to_csv(PSEUDOBULK_DIR / "test_8_pseudobulk_example_metrics.csv")


rmse       0.029869
pearson    0.961343
r2         0.923452
Name: metrics, dtype: float64

,celltype,rmse,pearson,r2
0,macrophages,0.012937,0.991764,0.983520
1,t_cells,0.013483,0.991239,0.981311
2,b_cells,0.013947,0.991373,0.981690
3,exc_neurons,0.017142,0.986925,0.972215
4,opc,0.017760,0.989494,0.976811
5,oligodendrocytes,0.018511,0.986831,0.969984
6,endothelial_cells,0.018620,0.988806,0.967917
7,inf_mg,0.024620,0.983165,0.957986
8,inh_neurons,0.025233,0.984000,0.950174
9,hom_mg,0.029253,0.970090,0.927047


,sample_id,rmse,pearson,r2
0,pb_01517,0.001893,0.999972,0.999945
1,pb_01843,0.003437,0.999901,0.999747
2,pb_00663,0.003826,0.999815,0.999623
3,pb_00399,0.004530,0.999765,0.999464
4,pb_01820,0.004649,0.999797,0.999458
...,...,...,...,...
2495,pb_00116,0.145530,0.811734,0.550049
2496,pb_01729,0.146851,0.858184,0.649773
2497,pb_00065,0.148999,0.842998,0.577143
2498,pb_00362,0.153186,0.700796,0.429558


,rmse,pearson,r2
sample_id,,,
pb_01517,0.001893,0.999972,0.999945
pb_01843,0.003437,0.999901,0.999747
pb_00663,0.003826,0.999815,0.999623
pb_00399,0.004530,0.999765,0.999464
pb_01820,0.004649,0.999797,0.999458
pb_02474,0.005083,0.999856,0.999348
pb_01758,0.005240,0.999743,0.999343
pb_02493,0.005398,0.999638,0.999261
pb_02258,0.137147,0.690700,0.476115


Saved stacked real-vs-predicted plot to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_std/pseudobulk/plots/test_8_pseudobulk_examples_real_vs_predicted_stacked.png
Saved best/worst celltype prediction plot to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_std/pseudobulk/plots/test_best_worst_celltype_predictions.png


,celltype,rmse,pearson,r2,group
0,macrophages,0.012937,0.991764,0.983520,best
1,t_cells,0.013483,0.991239,0.981311,best
2,b_cells,0.013947,0.991373,0.981690,best
3,prol_mg,0.032098,0.980674,0.900291,worst
4,epithelial_cells,0.044052,0.977787,0.815861,worst
5,fibroblasts,0.068166,0.982501,0.632960,worst


## 8. Save training artifacts

The checkpoint contains the model weights, training gene order, cell type order, and preprocessing statistics required to run predictions on new bulk RNA-seq samples.


In [43]:
def save_training_artifacts(
    model: MLPDeconvRegressor,
    output_dir: str | Path,
    config: TrainingConfig,
    genes: Sequence[str],
    cell_types: Sequence[str],
    gene_mean: np.ndarray,
    gene_std: np.ndarray,
    training_history: pd.DataFrame,
    metrics: pd.DataFrame,
    global_metrics: pd.Series,
) -> Path:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output_dir / "mlp_deconv_checkpoint.pt"
    checkpoint = {
        "state_dict": model.cpu().state_dict(),
        "model_config": asdict(config),
        "genes": list(genes),
        "cell_types": list(cell_types),
        "gene_mean": gene_mean.astype(np.float32),
        "gene_std": gene_std.astype(np.float32),
        "normalization": {"method": "CPM_log1p", "scale_factor": 1_000_000.0},
        "loss": config.loss_name,
    }
    torch.save(checkpoint, checkpoint_path)
    model.to(DEVICE)
    training_history.to_csv(output_dir / "training_history.csv", index=False)
    metrics.to_csv(output_dir / "test_metrics_by_celltype.csv", index=False)
    global_metrics.to_frame().to_csv(output_dir / "test_metrics_global.csv")
    print(f"Saved model checkpoint to {checkpoint_path}")
    return checkpoint_path


In [44]:
checkpoint_path = save_training_artifacts(
    model=model,
    output_dir=MODEL_DIR,
    config=training_config,
    genes=genes,
    cell_types=CELL_TYPES,
    gene_mean=gene_mean,
    gene_std=gene_std,
    training_history=training_history,
    metrics=per_celltype_metrics,
    global_metrics=global_metrics,
)
checkpoint_path


Saved model checkpoint to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_std/model/mlp_deconv_checkpoint.pt


PosixPath('/mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/references/rat_sub_std/model/mlp_deconv_checkpoint.pt')